# Reopen a Clerk case: Atera human kidney

Scratch notebook (not part of the docs) for browsing a Clerk case saved in `Clerk_Atera_Kidney_Landscape-Clustergram.ipynb`: its questions, answers and evidence, the cell types Clerk proposed, and the labels on the Clustergram (yours and Clerk's).

`dega.viz.Clerk(clerk_files=...)` reopens the case on fresh widgets showing the same data. **No Claude needed**:

- the panel shows the whole conversation; its genes, clusters, cited views and screenshots change the Landscape and Clustergram;
- the Clustergram shows the case's cell types (`cell_type` columns) and the marker genes behind them (`info` rows);
- each question's view is a pin on the Landscape (**PINS**): hover one for its question, click it to return to that view;
- a new question continues the case (that needs Claude), and new saves go beside it.

The last section copies the case and its Matrix into the docs for the unlisted gallery page `gallery/gallery_clerk_atera_kidney.md`.

In [ ]:
import os

# Use this checkout's JS build, served locally (restart the kernel after a rebuild).
os.environ["CELLDEGA_LOCAL_ESM"] = "serve"

In [ ]:
import json
from pathlib import Path
import shutil
from urllib.request import urlopen

import celldega as dega
import numpy as np

SAMPLE = "Atera_v1_WTA_Kidney_non_diseased_FFPE"
DATA_REVISION = "3a46ab4db67821eef3b8ff8d1c3704fb7b3bb48f"
BASE_URL = f"https://raw.githubusercontent.com/broadinstitute/Celldega_{SAMPLE}/{DATA_REVISION}"

# The saved case: a conversation from Save (.json), a report folder from Report, or
# a folder of them (clerk_files/atera_kidney opens the most recent).
CASE = Path("clerk_files/atera_kidney/conversations/conversation.json")

# The clustered Matrix the case was made with (written by the main kidney notebook).
cwd = Path.cwd().resolve()
repo = next(p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").is_file())
matrix_store = repo / "notebooks/data/atera_data" / SAMPLE / "kidney_leiden_marker_matrix.zarr"
if not matrix_store.exists():
    raise FileNotFoundError(
        f"{matrix_store} not found: run Clerk_Atera_Kidney_Landscape-Clustergram.ipynb once to compute it"
    )
matrix = dega.clust.Matrix.read_zarr(matrix_store)

with urlopen(f"{BASE_URL}/micron_to_image_transform.csv") as response:
    transform = np.loadtxt(response)

## The case, on its data

The widgets are made first and passed to Clerk, so the case's views become pins on the Landscape and its labels go on the Clustergram.

In [ ]:
landscape = dega.viz.Landscape(
    technology="Xenium", base_url=BASE_URL, transform=transform,
    dataset_name="Atera kidney — default clusters", height=600, max_tiles_to_view=25,
)
cgm = dega.viz.Clustergram(
    matrix=matrix, width=400, height=500, rank_dim=10,
    manual_col_cat="cell_type", manual_row_cat="info",
)
clerk = dega.viz.Clerk(clerk_files=CASE, landscape=landscape, clustergram=cgm)
print("Opened", clerk._case["path"], "saved", clerk._case.get("saved"))
print(len(landscape.pins), "pins;", len(clerk.findings), "proposals")

In [ ]:
linked = dega.viz.spatial_clustergram(landscape, cgm, width="500px", clerk=clerk)

In [ ]:
linked

## Cell types

One row per cluster labeled on the Clustergram or proposed by Clerk: the label and who set it, and Clerk's latest proposal.

In [ ]:
clerk.cell_types()

Every proposal, with its markers and the question it answered:

In [ ]:
clerk.findings_df()

Each question's views as Snapshot cards (e.g. `clerk.snapshots.card("Q1 Landscape")`):

In [ ]:
clerk.snapshots

## Copy the case to the docs (for testing the gallery page)

The unlisted gallery page `gallery/gallery_clerk_atera_kidney.md` reads the case and the Matrix in the browser, with no kernel. For now both are copied into `docs/assets/data/clerk_atera_kidney/`, which git ignores (any `data` folder is ignored), so they're served by `mkdocs serve` but never committed. Before the page is published, the Matrix needs a public home (e.g. the Hugging Face supporting-data repo).

The copy of the case leaves out what the page doesn't use and what points at this computer: the data `sources` (local paths) and the `record` (the full per-question log).

In [ ]:
docs_data = repo / "docs/assets/data/clerk_atera_kidney"
docs_data.mkdir(parents=True, exist_ok=True)

saved = json.loads(CASE.read_text(encoding="utf-8"))
gallery_case = {k: v for k, v in saved.items() if k not in ("sources", "record")}
(docs_data / "conversation.json").write_text(json.dumps(gallery_case), encoding="utf-8")

matrix_copy = docs_data / "kidney_leiden_markers.zarr"
if matrix_copy.exists():
    shutil.rmtree(matrix_copy)
shutil.copytree(matrix_store, matrix_copy)

for path in sorted(docs_data.iterdir()):
    size = sum(p.stat().st_size for p in path.rglob("*")) if path.is_dir() else path.stat().st_size
    print(f"{path.relative_to(repo)}  {size / 1e6:.1f} MB")